# Cholera Data Build — Bangladesh (WHO Global Dataset)

This notebook builds the base weekly dataset for the cholera outbreak prediction project.

## Goals
- Load raw WHO cholera datasets (2023–2024)
- Filter to Bangladesh
- Aggregate to weekly resolution
- Merge with weather data
- Save modeling-ready dataset

## Inputs

### Disease data
- Source: WHO Global Cholera Dashboard
- Files:
  - cholera_adm0_public_2023.csv
  - cholera_adm0_public_2024.csv

### Weather data
- Source: Open-Meteo (Dhaka)
- Variables:
  - temperature
  - precipitation
  - radiation

## Outputs

- cholera_weekly_merged.parquet
- cholera_weekly_merged.csv

## Notes
- Data is sparse and irregular
- Expect few outbreak weeks
- This dataset represents a low-structure system

In [2]:
from pathlib import Path
import sys
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(str(Path("..").resolve()))

from config import savefig, save_parquet, save_csv, print_config_summary

print_config_summary()

===== CONFIG SUMMARY =====
Base Dir: /Users/suvo/Projects/disease-outbreak
Default Disease: flu
Default Region: illinois
Forecast Horizons: [1, 2, 3, 4]
Default Lags: [1, 2, 3, 4]
Default Outbreak Quantile: 0.85
Raw Data Dir: /Users/suvo/Projects/disease-outbreak/data/raw
Interim Data Dir: /Users/suvo/Projects/disease-outbreak/data/interim
Processed Data Dir: /Users/suvo/Projects/disease-outbreak/data/processed
Figures Dir: /Users/suvo/Projects/disease-outbreak/outputs/figures
Tables Dir: /Users/suvo/Projects/disease-outbreak/outputs/tables
Models Dir: /Users/suvo/Projects/disease-outbreak/outputs/models
Flu Raw JSON: /Users/suvo/Projects/disease-outbreak/data/raw/flu/fluview_il_201040_202652.json
Flu Weather Raw JSON: /Users/suvo/Projects/disease-outbreak/data/raw/flu/weather_openmeteo_springfield_il_2010-10-01_2026-04-10.json


In [3]:
RAW_DIR = Path("../data/raw/cholera")

file_2023 = RAW_DIR / "cholera_adm0_public_2023.csv"
file_2024 = RAW_DIR / "cholera_adm0_public_2024.csv"
weather_path = RAW_DIR / "weather_openmeteo_dhaka_2023-01-01_2024-12-31.json"

print(file_2023.exists(), file_2024.exists(), weather_path.exists())

True True True


In [4]:
df_2023 = pd.read_csv(file_2023)
df_2024 = pd.read_csv(file_2024)

df = pd.concat([df_2023, df_2024], ignore_index=True)

print(df.shape)
display(df.head())

(64, 7)


,adm0_name,who_region,iso_3_code,first_epiwk,last_epiwk,case_total,death_total
0,AFGHANISTAN,Eastern Mediterranean Region,AFG,2023-01-02,2023-12-25,222221,97
1,BANGLADESH,South-East Asia Region,BGD,2023-01-02,2023-12-04,158,0
2,BURUNDI,African Region,BDI,2023-01-09,2023-12-25,1347,9
3,CAMEROON,African Region,CMR,2023-01-02,2023-12-25,6477,209
4,CONGO,African Region,COG,2023-07-24,2023-08-21,69,5


In [5]:
df.columns.tolist()

['adm0_name',
 'who_region',
 'iso_3_code',
 'first_epiwk',
 'last_epiwk',
 'case_total',
 'death_total']

In [8]:
df = df[df["adm0_name"] == "BANGLADESH"].copy()

print(df.shape)
display(df.head())

(2, 7)


,adm0_name,who_region,iso_3_code,first_epiwk,last_epiwk,case_total,death_total
1,BANGLADESH,South-East Asia Region,BGD,2023-01-02,2023-12-04,158,0
32,BANGLADESH,South-East Asia Region,BGD,2024-01-15,2024-12-23,575,0
